# META-CXR AUC-ROC + Optimal Threshold Figures (GCS checkpoint)

Notebook nay chay tren Kaggle de tao 2 nhom bieu do:

1. **AUC-ROC Curve for All Classes**: Positive, Negative, Uncertain vs rest cho 14 abnormality.
2. **Optimal Thresholds for Each Class and Abnormality**: threshold toi uu cho Positive, Negative, Uncertain.

Notebook clone source tu `https://github.com/minhphuong150505/Meta-CXR-Kaggle`, tai checkpoint tu Google Cloud Storage, chay inference tren split da chon, luu logits/probabilities va xuat hinh PNG/PDF trong `/kaggle/working`.

Mac dinh:
- checkpoint: `gs://meta-cxr-checkpoint/07_all_three/checkpoint_best.pth`
- split ve ROC: `test`
- split tune threshold: dung cung `EVAL_SPLIT` de ve giong figure mau. Neu threshold dung cho inference that, nen doi sang `val` de tranh tune tren test.

Yeu cau Kaggle:
- Bat Internet.
- Them Kaggle Secret `GCS_SERVICE_ACCOUNT` hoac `GCP_SERVICE_ACCOUNT_JSON` neu bucket private. Gia tri la service-account JSON co quyen doc bucket.
- Neu khong dung GCS secret, co the attach dataset checkpoint va set `LOCAL_CHECKPOINT_PATH` trong Cell 4.
- Attach dataset anh `mimic-cxr-jpg-lite` va dataset processed `mimic-cxr-p10-processed`, hoac mount theo layout `/kaggle/input/datasets/phuong20052/...`.


In [ ]:
# Cell 1 - Install dependencies for Kaggle.
# Pin transformers de khop Qformer/PubMedCLIP trong repo META-CXR.
import subprocess
import sys


def pip_install(*packages):
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "--upgrade-strategy", "only-if-needed", *packages,
        ],
        check=True,
    )


PACKAGES = [
    "omegaconf==2.3.0",
    "transformers==4.44.2",
    "opencv-python>=4.12,<4.13",
    "scikit-image>=0.22",
    "scikit-learn>=1.4",
    "pycocoevalcap",
    "torchinfo",
    "wandb",
    "loralib==0.1.1",
    "iterative-stratification",
    "iopath",
    "hi-ml-multimodal==0.2.2",
    "timm>=0.9.0",
    "spacy>=3.8,<3.9",
    "nltk>=3.9",
    "google-cloud-storage",
    "matplotlib",
    "seaborn",
]
pip_install(*PACKAGES)
pip_install("git+https://github.com/huggingface/peft.git@e536616888d51b453ed354a6f1e243fecb02ea08")

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

import numpy as _np
import pandas as _pd
import torch as _torch

print(f"numpy  = {_np.__version__}")
print(f"pandas = {_pd.__version__}")
print(f"torch  = {_torch.__version__}")
print(f"GPUs   = {_torch.cuda.device_count()}")
for _i in range(_torch.cuda.device_count()):
    print(f"  GPU {_i}: {_torch.cuda.get_device_name(_i)}")


In [ ]:
# Cell 2 - Clone META-CXR source from GitHub.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/minhphuong150505/Meta-CXR-Kaggle.git"
PROJECT_DIR = Path("/kaggle/working/META-CXR")

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")


In [ ]:
# Cell 3 - Configure Kaggle data paths and env_config.yaml for local_config.py.
import os
import shutil
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")
OWNER_DATASET_ROOT = INPUT_DIR / "datasets" / "phuong20052"


def is_meta_cxr_project(path: Path) -> bool:
    return (path / "model" / "lavis").exists() and (path / "pretraining").exists()


def _candidate_roots(base: Path):
    yield base
    versions = base / "versions"
    if versions.exists():
        for version_dir in sorted(versions.iterdir()):
            if version_dir.is_dir():
                yield version_dir


def find_dataset_root(name_candidates, required_files):
    search_bases = []
    for name in name_candidates:
        search_bases.extend([
            OWNER_DATASET_ROOT / name,
            INPUT_DIR / "datasets" / name,
            INPUT_DIR / name,
        ])

    for root in [OWNER_DATASET_ROOT, INPUT_DIR / "datasets", INPUT_DIR]:
        if root.exists():
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    search_bases.append(child)

    checked = []
    seen = set()
    for base in search_bases:
        for candidate in _candidate_roots(Path(base)):
            key = str(candidate)
            if key in seen:
                continue
            seen.add(key)
            checked.append(key)
            if candidate.exists() and all((candidate / rel).exists() for rel in required_files):
                return candidate

    raise FileNotFoundError(
        "Khong tim thay dataset co du files " + str(required_files) + "\nDa kiem tra:\n" + "\n".join(checked[:80])
    )


if not is_meta_cxr_project(PROJECT_DIR):
    raise FileNotFoundError(f"{PROJECT_DIR} khong phai META-CXR project.")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / "model"))

IMAGE_ROOT = find_dataset_root(
    ["mimic-cxr-jpg-lite", "MIMIC-CXR-JPG-LITE"],
    ["mimic-cxr-2.0.0-split.csv", "mimic-cxr-2.0.0-chexpert.csv", "mimic-cxr-2.0.0-metadata.csv"],
)
PROCESSED_ROOT = find_dataset_root(
    ["mimic-cxr-p10-processed", "mimic-cxr-p10-preprocessed", "MIMIC-CXR p10 Preprocessed"],
    ["train.csv", "val.csv", "test.csv"],
)

CHECKPOINT_ROOT = WORK_DIR / "checkpoints"
OUTPUT_DIR = WORK_DIR / "figures"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JAVA_HOME = "/usr/lib/jvm/java-11-openjdk-amd64"
JAVA_PATH = JAVA_HOME + "/bin:"

(PROJECT_DIR / "configs").mkdir(exist_ok=True)
(PROJECT_DIR / "configs" / "env_config.yaml").write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "{OUTPUT_DIR}"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
wandb:
  entity: ""
  project: "meta-cxr-encoder-comparison"
java:
  home: "{JAVA_HOME}"
  path: "{JAVA_PATH}"
''', encoding="utf-8")

print("PROJECT_DIR    =", PROJECT_DIR)
print("IMAGE_ROOT     =", IMAGE_ROOT)
print("PROCESSED_ROOT =", PROCESSED_ROOT)
print("CHECKPOINT_ROOT=", CHECKPOINT_ROOT)
print("OUTPUT_DIR     =", OUTPUT_DIR)


In [ ]:
# Cell 4 - Choose checkpoint/run and evaluation settings.
# Doi RUN_NAME neu muon ve figure cho encoder combo khac.
RUN_NAME = "07_all_three"

# Bucket/path mac dinh: gs://meta-cxr-checkpoint/<RUN_NAME>/checkpoint_best.pth
GCS_BUCKET_NAME = "meta-cxr-checkpoint"
CHECKPOINT_FILENAME = "checkpoint_best.pth"

# Optional: neu attach checkpoint bang Kaggle Dataset, dien path .pth o day de bo qua GCS.
# Vi du: LOCAL_CHECKPOINT_PATH = "/kaggle/input/my-checkpoints/07_all_three/checkpoint_best.pth"
LOCAL_CHECKPOINT_PATH = None

# Split dung de ve ROC va threshold figure.
# Neu threshold dung cho inference/bao cao that, nen doi sang "val" de khong tune tren test.
EVAL_SPLIT = "test"

EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2

# Set None de chay toan bo split. Co the dat 128/256 de smoke test nhanh.
SAMPLE_LIMIT = None

# Neu True, classification head nhan text embeddings tu report ground truth,
# giong notebook threshold/table mau trong repo. False dung image-only forward_image,
# phu hop inference thuc te hon.
USE_REPORT_TEXT_FOR_CLASSIFICATION = True


In [ ]:
# Cell 5 - GCS authentication and checkpoint download.
import base64
import json
import os
from pathlib import Path

import google.auth
from google.api_core.exceptions import Forbidden, GoogleAPICallError, NotFound, Unauthorized
from google.auth.exceptions import DefaultCredentialsError, RefreshError
from google.cloud import storage
from google.oauth2 import service_account

try:
    from google.auth.exceptions import InvalidOperation as GoogleInvalidOperation
except Exception:
    GoogleInvalidOperation = RuntimeError


GCS_BUCKET_NAME = str(GCS_BUCKET_NAME).replace("gs://", "").rstrip("/")

GCS_AUTH_HELP = f'''Cannot access gs://{GCS_BUCKET_NAME}/{RUN_NAME}/{CHECKPOINT_FILENAME}.
Fix one of these before rerunning Cell 5:
1. Kaggle Secrets: add GCS_SERVICE_ACCOUNT or GCP_SERVICE_ACCOUNT_JSON containing the full service-account JSON.
2. Or attach a Kaggle Dataset containing the checkpoint and set LOCAL_CHECKPOINT_PATH in Cell 4.
3. The service account must have Storage Object Viewer permission on bucket {GCS_BUCKET_NAME}.'''


def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def _parse_json_or_base64(raw):
    raw = raw.strip()
    try:
        value = json.loads(raw)
    except json.JSONDecodeError:
        value = json.loads(base64.b64decode(raw).decode("utf-8"))
    if isinstance(value, str):
        value = json.loads(value)
    if not isinstance(value, dict) or "client_email" not in value or "private_key" not in value:
        raise ValueError("Service-account secret is not a valid JSON key.")
    return value


def load_service_account_info():
    for name in ("GCS_SERVICE_ACCOUNT", "GCP_SERVICE_ACCOUNT_JSON", "GCP_SERVICE_ACCOUNT_B64"):
        raw = get_secret(name)
        if raw:
            info = _parse_json_or_base64(raw)
            print(f"Using Kaggle/env secret: {name} ({info.get('client_email')})")
            return info
    return None


def build_storage_client():
    info = load_service_account_info()
    if info:
        credentials = service_account.Credentials.from_service_account_info(info)
        return storage.Client(project=info.get("project_id"), credentials=credentials)

    adc_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
    if adc_path and Path(adc_path).exists():
        print(f"Using GOOGLE_APPLICATION_CREDENTIALS={adc_path}")
        return storage.Client()

    try:
        credentials, project_id = google.auth.default()
        credentials_name = credentials.__class__.__name__.lower()
        if "anonymous" in credentials_name:
            raise DefaultCredentialsError("Default credentials are anonymous.")
        print(f"Using application default credentials: {credentials.__class__.__name__}")
        return storage.Client(project=project_id, credentials=credentials)
    except Exception as exc:
        raise RuntimeError(GCS_AUTH_HELP) from exc


def local_checkpoint_candidates(run_name, filename):
    candidates = []
    if LOCAL_CHECKPOINT_PATH:
        candidates.append(Path(LOCAL_CHECKPOINT_PATH))

    candidates.extend([
        CHECKPOINT_ROOT / run_name / filename,
        PROJECT_DIR / "pretraining" / "output" / run_name / filename,
        PROJECT_DIR / "pretraining" / "output" / filename,
    ])

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for root in sorted(input_root.iterdir()):
            if not root.is_dir():
                continue
            candidates.extend([
                root / run_name / filename,
                root / filename,
                root / "checkpoints" / run_name / filename,
                root / "pretraining" / "output" / run_name / filename,
            ])
            versions = root / "versions"
            if versions.exists():
                for version_dir in sorted(versions.iterdir()):
                    if version_dir.is_dir():
                        candidates.extend([
                            version_dir / run_name / filename,
                            version_dir / filename,
                            version_dir / "checkpoints" / run_name / filename,
                            version_dir / "pretraining" / "output" / run_name / filename,
                        ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists() and candidate.is_file():
            yield candidate


def find_local_checkpoint(run_name, filename):
    found = list(local_checkpoint_candidates(run_name, filename))
    if not found:
        return None
    chosen = sorted(found, key=lambda x: (len(str(x)), str(x)))[0]
    print(f"Using local checkpoint: {chosen}")
    return chosen


def _download_exact_blob(client, blob_name, local_path):
    bucket = client.bucket(GCS_BUCKET_NAME)
    blob = bucket.blob(blob_name)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        print(f"Downloading gs://{GCS_BUCKET_NAME}/{blob_name} -> {local_path}")
        blob.download_to_filename(str(local_path))
        print(f"Downloaded {local_path.stat().st_size / 1e6:.1f} MB")
        return True
    except NotFound:
        return False
    except (Forbidden, Unauthorized, RefreshError, DefaultCredentialsError, GoogleInvalidOperation) as exc:
        raise RuntimeError(GCS_AUTH_HELP) from exc
    except GoogleAPICallError as exc:
        raise RuntimeError(f"GCS API error while downloading {blob_name}: {exc}") from exc


def _find_nested_blob(client, run_name, filename):
    try:
        candidates = [
            blob for blob in client.list_blobs(GCS_BUCKET_NAME, prefix=f"{run_name}/")
            if blob.name == f"{run_name}/{filename}" or blob.name.endswith("/" + filename)
        ]
    except (Forbidden, Unauthorized, RefreshError, DefaultCredentialsError, GoogleInvalidOperation) as exc:
        raise RuntimeError(GCS_AUTH_HELP) from exc
    except GoogleAPICallError as exc:
        raise RuntimeError(f"GCS API error while listing {run_name}: {exc}") from exc

    if not candidates:
        return None
    return sorted(candidates, key=lambda b: ((b.updated.timestamp() if b.updated else 0), b.name))[-1]


def download_checkpoint_from_gcs(run_name, filename):
    client = build_storage_client()
    local = CHECKPOINT_ROOT / run_name / filename
    exact_blob_name = f"{run_name}/{filename}"

    if _download_exact_blob(client, exact_blob_name, local):
        return local

    nested_blob = _find_nested_blob(client, run_name, filename)
    if nested_blob is None:
        return None

    local = CHECKPOINT_ROOT / nested_blob.name
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        print(f"Downloading gs://{GCS_BUCKET_NAME}/{nested_blob.name} -> {local}")
        nested_blob.download_to_filename(str(local))
        print(f"Downloaded {local.stat().st_size / 1e6:.1f} MB")
        return local
    except (Forbidden, Unauthorized, RefreshError, DefaultCredentialsError, GoogleInvalidOperation) as exc:
        raise RuntimeError(GCS_AUTH_HELP) from exc
    except GoogleAPICallError as exc:
        raise RuntimeError(f"GCS API error while downloading {nested_blob.name}: {exc}") from exc


def ensure_local_checkpoint(run_name, filename=CHECKPOINT_FILENAME):
    local = find_local_checkpoint(run_name, filename)
    if local is not None:
        return local

    downloaded = download_checkpoint_from_gcs(run_name, filename)
    if downloaded is not None:
        return downloaded

    if filename != "checkpoint_last.pth":
        print(f"{filename} not found; trying checkpoint_last.pth")
        return ensure_local_checkpoint(run_name, "checkpoint_last.pth")

    raise FileNotFoundError(f"No {filename} found locally or in gs://{GCS_BUCKET_NAME}/{run_name}/")


checkpoint_path = ensure_local_checkpoint(RUN_NAME)
checkpoint_path


In [ ]:
# Cell 6 - Imports, constants, model loader, and data loader.
import gc
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import auc, roc_curve
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping["paths"]["cache_root"] = "."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ABNORMALITIES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices",
]
CLASS_MAP = {"negative": 0, "positive": 1, "uncertain": 2}
CLASS_PLOT_ORDER = [
    ("positive", "Positive Class vs Rest"),
    ("negative", "Negative Class vs Rest"),
    ("uncertain", "Uncertain Class vs Rest"),
]


def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / "pretraining" / "configs" / "encoder_comparison" / f"{run_name}.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Khong tim thay config: {cfg_path}")
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)


def load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def filter_state_dict_for_model(model, state_dict):
    model_state = model.state_dict()
    filtered = {}
    mismatched = []
    for key, value in state_dict.items():
        if key in model_state and hasattr(value, "shape"):
            ckpt_shape = tuple(value.shape)
            model_shape = tuple(model_state[key].shape)
            if ckpt_shape != model_shape:
                mismatched.append((key, ckpt_shape, model_shape))
                continue
        filtered[key] = value
    return filtered, mismatched


def build_model_for_run(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = load_torch_checkpoint(checkpoint_path)
    state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    state_dict, mismatched = filter_state_dict_for_model(model, state_dict)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(
        f"{run_name}: loaded {checkpoint_path.name}; "
        f"missing={len(missing)}, unexpected={len(unexpected)}, mismatched_skipped={len(mismatched)}"
    )
    if mismatched:
        for key, ckpt_shape, model_shape in mismatched[:8]:
            print(f"  skipped shape mismatch: {key}: checkpoint={ckpt_shape}, model={model_shape}")
    model.to(DEVICE)
    model.eval()
    return cfg, model


def make_loader(cfg, split: str):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split=split,
        cfg=cfg,
        truncate=SAMPLE_LIMIT,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == "cuda"),
    )


@torch.no_grad()
def predict_logits_with_text(model, batch):
    image = batch["image"].to(DEVICE, non_blocking=True)
    text = batch["text_output"]
    cnn_patches, vit_patches, swin_patches, _ = model._encode_image_streams(image, apply_aug=False)
    text_tokens = model.tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=model.max_txt_len,
        return_tensors="pt",
    ).to(DEVICE)
    text_output = model.Qformer.bert(
        text_tokens.input_ids,
        attention_mask=text_tokens.attention_mask,
        return_dict=True,
    )
    logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches,
        vit_patches=vit_patches,
        swin_patches=swin_patches,
        text_embeddings=text_output.last_hidden_state,
        labels=None,
    )
    return logits


@torch.no_grad()
def predict_logits_image_only(model, batch):
    image = batch["image"].to(DEVICE, non_blocking=True)
    logits, _ = model.forward_image(image)
    return logits


print("DEVICE =", DEVICE)


In [ ]:
# Cell 7 - Run inference and save probabilities/labels.
cfg, model = build_model_for_run(RUN_NAME, checkpoint_path)
loader = make_loader(cfg, EVAL_SPLIT)

all_probs = []
all_labels = []
all_dicom_ids = []
predict_fn = predict_logits_with_text if USE_REPORT_TEXT_FOR_CLASSIFICATION else predict_logits_image_only

for batch in tqdm(loader, desc=f"{RUN_NAME}:{EVAL_SPLIT}"):
    logits = predict_fn(model, batch)
    probs_batch = torch.softmax(logits, dim=-1)
    all_probs.append(probs_batch.detach().cpu().numpy())
    all_labels.append(batch["classification_labels"].detach().cpu().numpy())
    all_dicom_ids.extend([str(x) for x in batch["dicom_id"]])

probs = np.concatenate(all_probs, axis=0)
labels = np.concatenate(all_labels, axis=0)

npz_path = WORK_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_classification_probs.npz"
np.savez_compressed(
    npz_path,
    probs=probs,
    labels=labels,
    dicom_ids=np.array(all_dicom_ids),
    abnormalities=np.array(ABNORMALITIES),
)
print("Saved", npz_path)
print("probs shape =", probs.shape)
print("labels shape =", labels.shape)

del model, loader
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# Cell 8 - Compute ROC curves, AUC, and optimal thresholds by Youden J.
import json

def binary_roc_for_class(labels, probs, abnormality_idx, class_idx):
    y_true = (labels[:, abnormality_idx] == class_idx).astype(int)
    scores = probs[:, abnormality_idx, class_idx]
    positive_count = int(y_true.sum())
    negative_count = int(len(y_true) - positive_count)
    if positive_count == 0 or negative_count == 0:
        return None
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    roc_auc = float(auc(fpr, tpr))

    finite = np.isfinite(thresholds)
    if finite.any():
        fpr_f = fpr[finite]
        tpr_f = tpr[finite]
        thresholds_f = thresholds[finite]
        best_idx = int(np.argmax(tpr_f - fpr_f))
        best_threshold = float(thresholds_f[best_idx])
        best_youden_j = float(tpr_f[best_idx] - fpr_f[best_idx])
    else:
        best_threshold = float("nan")
        best_youden_j = float("nan")

    return {
        "fpr": fpr,
        "tpr": tpr,
        "thresholds": thresholds,
        "auc": roc_auc,
        "optimal_threshold": best_threshold,
        "youden_j": best_youden_j,
        "positive_count": positive_count,
        "negative_count": negative_count,
    }


roc_results = {class_name: {} for class_name in CLASS_MAP}
threshold_rows = []
threshold_json = {}

for abn_idx, abnormality in enumerate(ABNORMALITIES):
    threshold_json[abnormality] = {}
    for class_name, class_idx in CLASS_MAP.items():
        result = binary_roc_for_class(labels, probs, abn_idx, class_idx)
        if result is None:
            continue
        roc_results[class_name][abnormality] = result
        threshold_json[abnormality][class_name] = result["optimal_threshold"]
        threshold_rows.append({
            "abnormality": abnormality,
            "class": class_name,
            "auc": result["auc"],
            "optimal_threshold": result["optimal_threshold"],
            "youden_j": result["youden_j"],
            "positive_count": result["positive_count"],
            "negative_count": result["negative_count"],
            "n": result["positive_count"] + result["negative_count"],
        })

threshold_df = pd.DataFrame(threshold_rows)
threshold_csv = WORK_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_optimal_thresholds.csv"
threshold_json_path = WORK_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_optimal_thresholds.json"
threshold_df.to_csv(threshold_csv, index=False)
threshold_json_path.write_text(json.dumps(threshold_json, indent=4), encoding="utf-8")

print("Saved", threshold_csv)
print("Saved", threshold_json_path)
display(threshold_df.head(20))


In [ ]:
# Cell 9 - Plot Image #1 style: AUC-ROC curves for all classes.
def plot_roc_curves(roc_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle("AUC-ROC Curve for All Classes", fontsize=16, y=1.02)

    for ax, (class_name, title) in zip(axes, CLASS_PLOT_ORDER):
        for abnormality in ABNORMALITIES:
            result = roc_results[class_name].get(abnormality)
            if result is None:
                continue
            ax.plot(
                result["fpr"],
                result["tpr"],
                linewidth=1.6,
                label=f"{abnormality} (AUC = {result['auc']:.2f})",
            )
        ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.0, alpha=0.6)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("False Positive Rate")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.35)
        ax.legend(fontsize=8, loc="lower right", frameon=True)

    axes[0].set_ylabel("True Positive Rate")
    fig.tight_layout()
    return fig


roc_fig = plot_roc_curves(roc_results)
roc_png = OUTPUT_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_auc_roc_all_classes.png"
roc_pdf = OUTPUT_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_auc_roc_all_classes.pdf"
roc_fig.savefig(roc_png, dpi=300, bbox_inches="tight")
roc_fig.savefig(roc_pdf, bbox_inches="tight")
print("Saved", roc_png)
print("Saved", roc_pdf)
plt.show()


In [ ]:
# Cell 10 - Plot Image #2 style: optimal thresholds for each class and abnormality.
def thresholds_for_class(class_name):
    values = []
    names = []
    for abnormality in ABNORMALITIES:
        result = roc_results[class_name].get(abnormality)
        if result is None:
            continue
        names.append(abnormality)
        values.append(result["optimal_threshold"])
    return names, values


def plot_optimal_thresholds(roc_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle("Optimal Thresholds for Each Class and Abnormality", fontsize=16, y=1.02)

    for ax, (class_name, title) in zip(axes, CLASS_PLOT_ORDER):
        names, values = thresholds_for_class(class_name)
        ax.bar(names, values, color="#2c98c9", edgecolor="white", linewidth=0.6)
        ax.set_title("Optimal Thresholds: " + title.replace(" vs Rest", ""), fontsize=12)
        ax.set_xlabel("Abnormality")
        ax.set_ylim(0, 1)
        ax.grid(axis="y", alpha=0.35)
        ax.tick_params(axis="x", labelrotation=50)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")

    axes[0].set_ylabel("Threshold")
    fig.tight_layout()
    return fig


threshold_fig = plot_optimal_thresholds(roc_results)
threshold_png = OUTPUT_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_optimal_thresholds.png"
threshold_pdf = OUTPUT_DIR / f"{RUN_NAME}_{EVAL_SPLIT}_optimal_thresholds.pdf"
threshold_fig.savefig(threshold_png, dpi=300, bbox_inches="tight")
threshold_fig.savefig(threshold_pdf, bbox_inches="tight")
print("Saved", threshold_png)
print("Saved", threshold_pdf)
plt.show()


## Files generated

Sau khi chay het notebook:

- `/kaggle/working/figures/<RUN_NAME>_<EVAL_SPLIT>_auc_roc_all_classes.png`
- `/kaggle/working/figures/<RUN_NAME>_<EVAL_SPLIT>_auc_roc_all_classes.pdf`
- `/kaggle/working/figures/<RUN_NAME>_<EVAL_SPLIT>_optimal_thresholds.png`
- `/kaggle/working/figures/<RUN_NAME>_<EVAL_SPLIT>_optimal_thresholds.pdf`
- `/kaggle/working/<RUN_NAME>_<EVAL_SPLIT>_classification_probs.npz`
- `/kaggle/working/<RUN_NAME>_<EVAL_SPLIT>_optimal_thresholds.csv`
- `/kaggle/working/<RUN_NAME>_<EVAL_SPLIT>_optimal_thresholds.json`
